# Exploratory Data Analysis — Performance Test Classification

This notebook explores the Locust performance test data to understand distributions,
class balance, transaction patterns, and feature characteristics before model training.

**Data source:** Postgres database via `src.extract` (or CSV fallback for offline use)

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 100)

print('Imports ready')

## 1. Data Loading

Try loading from the database first. If no `.env` is configured, fall back to
the CSV sample files in `database_samples/` for offline exploration.

In [ ]:
try:
    from src.extract import extract_training_data
    df_raw = extract_training_data()
    data_source = 'database'
except Exception as e:
    print(f'DB connection failed ({e}), loading from CSV samples...')
    ts = pd.read_csv('../database_samples/test_summary.csv')
    tr = pd.read_csv('../database_samples/testrun.csv')
    # Simulate the JOIN
    df_raw = ts.merge(
        tr[['id', 'testplan', 'exit_code', 'num_clients', 'rps_avg',
            'resp_time_avg', 'fail_ratio', 'build_version', 'requests']],
        left_on=['testplan', 'run_id'],
        right_on=['testplan', 'id'],
        how='inner'
    )
    df_raw = df_raw.rename(columns={
        'name': 'transaction_name',
        'requests_x': 'txn_requests',
        'failed': 'txn_failed',
        'requests_y': 'total_requests',
    })
    # Apply exclusion filters
    df_raw = df_raw[
        df_raw['exit_code'].notna() &
        (df_raw['exit_code'] != 0) &
        (df_raw['rps_avg'] > 0) &
        (df_raw['num_clients'] > 1)
    ].copy()
    # Exclude SignalR_Analysis meta-row
    df_raw = df_raw[df_raw['transaction_name'] != 'SignalR_Analysis']
    data_source = 'csv_samples'

print(f'Data source: {data_source}')
print(f'Shape: {df_raw.shape}')
print(f'Distinct runs: {df_raw["testplan"].nunique()}')

## 2. Data Overview

In [ ]:
print('=== Shape ===')
print(f'Rows: {len(df_raw):,}  Columns: {len(df_raw.columns)}')
print(f'\n=== Columns & Types ===')
print(df_raw.dtypes)
print(f'\n=== Null Counts ===')
print(df_raw.isnull().sum())
print(f'\n=== Distinct Counts ===')
print(f'Distinct testplans (runs): {df_raw["testplan"].nunique()}')
print(f'Distinct transaction names: {df_raw["transaction_name"].nunique()}')

In [ ]:
df_raw.describe()

## 3. Class Distribution (Exit Code)

In [ ]:
# One row per run for class analysis
runs = df_raw.drop_duplicates('testplan')[['testplan', 'exit_code', 'num_clients', 'build_version']].copy()
runs['label'] = runs['exit_code'].map({1: 'Pass', 2: 'Fail (TPS)', 3: 'Fail (RT)', 4: 'Fail (Error)'})

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Exit code distribution
exit_counts = runs['exit_code'].value_counts().sort_index()
colors = {1: '#2ecc71', 2: '#e67e22', 3: '#e74c3c', 4: '#9b59b6'}
bar_colors = [colors.get(ec, '#95a5a6') for ec in exit_counts.index]
axes[0].bar(exit_counts.index.astype(str), exit_counts.values, color=bar_colors)
axes[0].set_xlabel('Exit Code')
axes[0].set_ylabel('Number of Runs')
axes[0].set_title('Exit Code Distribution')
for i, (ec, count) in enumerate(exit_counts.items()):
    axes[0].text(i, count + 0.5, str(count), ha='center', fontweight='bold')

# Pass vs Fail
pass_fail = runs['exit_code'].apply(lambda x: 'Pass' if x == 1 else 'Fail').value_counts()
axes[1].pie(pass_fail, labels=pass_fail.index, autopct='%1.1f%%',
            colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[1].set_title('Pass vs Fail')

plt.tight_layout()
plt.show()

print(f'\nPass/Fail ratio: {(runs["exit_code"]==1).sum()}:{(runs["exit_code"]!=1).sum()}')

## 4. Test Type Distribution

In [ ]:
from src.features import derive_test_type

runs['test_type'] = runs['build_version'].apply(derive_test_type)

fig, ax = plt.subplots(figsize=(8, 4))
tt_counts = runs['test_type'].value_counts()
tt_counts.plot.bar(ax=ax, color='steelblue')
ax.set_title('Runs by Test Type')
ax.set_ylabel('Count')
ax.set_xlabel('Test Type')
for i, v in enumerate(tt_counts.values):
    ax.text(i, v + 0.3, str(v), ha='center', fontweight='bold')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(f'\nUser count distribution:')
print(runs['num_clients'].value_counts().to_string())

## 5. Transaction Analysis

In [ ]:
# Transactions per run
txn_per_run = df_raw.groupby('testplan')['transaction_name'].nunique()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of txn count per run
axes[0].hist(txn_per_run, bins=20, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Number of Transactions')
axes[0].set_ylabel('Number of Runs')
axes[0].set_title('Transactions per Test Run')
axes[0].axvline(txn_per_run.median(), color='red', linestyle='--', label=f'median={txn_per_run.median():.0f}')
axes[0].legend()

# Most common transactions
top_txns = df_raw['transaction_name'].value_counts().head(15)
top_txns.plot.barh(ax=axes[1], color='steelblue')
axes[1].set_xlabel('Occurrences Across Runs')
axes[1].set_title('Top 15 Most Common Transactions')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print(f'\nMedian transactions per run: {txn_per_run.median():.0f}')
print(f'Passing runs median: {txn_per_run[txn_per_run.index.isin(runs[runs["exit_code"]==1]["testplan"])].median():.0f}')
print(f'Failing runs median: {txn_per_run[txn_per_run.index.isin(runs[runs["exit_code"]!=1]["testplan"])].median():.0f}')

## 6. Response Time Analysis

In [ ]:
# Tag each row with pass/fail for visualization
df_viz = df_raw.copy()
df_viz['status'] = df_viz['exit_code'].apply(lambda x: 'Pass' if x == 1 else 'Fail')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# P95 distribution by status (log scale for readability)
for status, color in [('Pass', '#2ecc71'), ('Fail', '#e74c3c')]:
    subset = df_viz[df_viz['status'] == status]['perc_95'].clip(upper=df_viz['perc_95'].quantile(0.99))
    axes[0].hist(subset, bins=50, alpha=0.6, label=status, color=color)
axes[0].set_xlabel('P95 Response Time (ms)')
axes[0].set_ylabel('Transaction Count')
axes[0].set_title('P95 Distribution: Pass vs Fail')
axes[0].legend()
axes[0].set_yscale('log')

# Avg response time by status
for status, color in [('Pass', '#2ecc71'), ('Fail', '#e74c3c')]:
    subset = df_viz[df_viz['status'] == status]['avg_response_time'].clip(upper=df_viz['avg_response_time'].quantile(0.99))
    axes[1].hist(subset, bins=50, alpha=0.6, label=status, color=color)
axes[1].set_xlabel('Avg Response Time (ms)')
axes[1].set_ylabel('Transaction Count')
axes[1].set_title('Avg RT Distribution: Pass vs Fail')
axes[1].legend()
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

In [ ]:
# Run-level metrics comparison
run_metrics = df_raw.drop_duplicates('testplan')[['testplan', 'exit_code', 'rps_avg', 'resp_time_avg', 'fail_ratio']].copy()
run_metrics['status'] = run_metrics['exit_code'].apply(lambda x: 'Pass' if x == 1 else 'Fail')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.boxplot(data=run_metrics, x='status', y='rps_avg', ax=axes[0], palette={'Pass': '#2ecc71', 'Fail': '#e74c3c'})
axes[0].set_title('RPS Avg by Status')

sns.boxplot(data=run_metrics, x='status', y='resp_time_avg', ax=axes[1], palette={'Pass': '#2ecc71', 'Fail': '#e74c3c'})
axes[1].set_title('Resp Time Avg by Status')
axes[1].set_yscale('log')

sns.boxplot(data=run_metrics, x='status', y='fail_ratio', ax=axes[2], palette={'Pass': '#2ecc71', 'Fail': '#e74c3c'})
axes[2].set_title('Fail Ratio by Status')
axes[2].set_yscale('log')

plt.tight_layout()
plt.show()

## 7. Feature Engineering & Correlation

In [ ]:
from src.features import build_features, MODEL_FEATURES

run_df, baselines = build_features(df_raw, is_training=True)
print(f'\nRun-level features shape: {run_df.shape}')
run_df[MODEL_FEATURES + ['label_pass_fail', 'testplan']].head(10)

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(12, 10))
corr = run_df[MODEL_FEATURES + ['label_pass_fail']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, vmin=-1, vmax=1)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 8. Class Balance Assessment

In [ ]:
n_pass = (run_df['label_pass_fail'] == 1).sum()
n_fail = (run_df['label_pass_fail'] == 0).sum()
total = n_pass + n_fail
fail_pct = n_fail / total * 100

print(f'Pass: {n_pass} ({n_pass/total*100:.1f}%)')
print(f'Fail: {n_fail} ({fail_pct:.1f}%)')
print(f'Ratio: {n_pass}:{n_fail} ({n_pass/max(n_fail,1):.1f}:1)')
print()
if fail_pct < 15:
    print('⚠️  Class imbalance detected (fail < 15%).')
    print('   Recommended: use class_weight="balanced" during training.')
    print('   Consider SMOTE if balanced weighting is insufficient.')
else:
    print('✅ Class distribution is reasonably balanced (fail >= 15%).')
    print('   class_weight="balanced" still recommended as a precaution.')

## 9. Summary

**Key observations from EDA:**

1. **Class distribution:** Documented above — if imbalanced, `class_weight='balanced'` will be used
2. **Transaction patterns:** Passing runs have ~55 transactions; failing runs balloon to ~78-80 due to SignalR command timeouts
3. **Response time separation:** Clear separation between pass/fail runs in p95 and avg RT — good signal for the model
4. **Feature correlations:** The % deviation features should show strong correlation with the label
5. **Baseline feasibility:** Enough passing runs to compute stable per-transaction medians

**Next step:** Proceed to model training with `python -m src.train`